In [ ]:
#BİR VERİ SETİ İÇİNDE  LOG-LİKEHOOD OLASILIKLARINI İNCELEYEN CODE MODELİ

In [1]:
!pip install hmmlearn

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 166.0/166.0 kB 10.8 MB/s eta 0:00:00


In [21]:
#KULLANILAN KÜTÜPHANELER
import pandas as pd
import numpy as np
from hmmlearn import hmm
from google.colab import files

In [22]:
from google.colab import files

# CSV dosyasını bilgisayardan Colab'a yüklemek için
uploaded = files.upload()
df= pd.read_csv("veri_seti.csv")

Saving veri_seti.csv to veri_seti (3).csv


In [23]:
#EV MODELİ
#high=1 low=0
model_ev = hmm.MultinomialHMM(n_components=2, n_trials=1) #e ve v diye iki state var
model_ev.startprob_ = np.array([1.0, 0.0])#başlangiç durumu e=1 v=0

model_ev.transmat_ = np.array([
    [0.6, 0.4],
    [0.2, 0.8]
])#geçiş matrisleri


model_ev.emissionprob_ = np.array([
    [0.7, 0.3],
    [0.1, 0.9]
])# gözlem aşaması


https://github.com/hmmlearn/hmmlearn/issues/335
https://github.com/hmmlearn/hmmlearn/issues/340


In [24]:
#OKUL MODELİ
model_okul = hmm.MultinomialHMM(n_components=4, n_trials=1) # O K U L 4 state var
model_okul.startprob_ = np.array([1.0, 0.0, 0.0, 0.0]) #o=1 diğerleri sıfır durumundan başlanılsın
model_okul.transmat_ = np.array([
    [0.7, 0.3, 0.0, 0.0],
    [0.2, 0.6, 0.2, 0.0],
    [0.0, 0.3, 0.7, 0.0],
    [0.0, 0.0, 0.0, 1.0]
])
model_okul.emissionprob_ = np.array([
    [0.8, 0.2],
    [0.3, 0.7],
    [0.5, 0.5],
    [0.2, 0.8]
])


https://github.com/hmmlearn/hmmlearn/issues/335
https://github.com/hmmlearn/hmmlearn/issues/340


In [25]:
#CSV DEKİ VERİLERİ HMM'YE UYGUN FORMATA GETİRME
df["gözlemler_list"] = df["gözlemler"].apply(lambda x: [int(i) for i in x.split()])
#gözlemler bölümü string olarak "0110" önce tek tek ayırıyoruz(split()ile) ardından her birini integera çeviriyoruz.
#lambda:ananom küçük fonskiyon
#apply() :pandastaki bir sütundaki her değere bir fonksiyon uygulamak için kullan
df["onehot"] = df["gözlemler_list"].apply(lambda obs: np.array([[1,0] if o==0 else [0,1] for o in obs]))


In [26]:
#LOG-LİKEHOOD'U HESAPLAYAN FONKSİYON
def classify_word_from_df(onehot_data):
    score_ev = model_ev.score(onehot_data)
    score_okul = model_okul.score(onehot_data)
    tahmin = "EV" if score_ev > score_okul else "OKUL"
    return tahmin, score_ev, score_okul


In [27]:
# SONUCLAR CSV DOSYASI OLUŞTURULDU
df["tahmin_model"], df["score_ev"], df["score_okul"] = zip(*df["onehot"].apply(classify_word_from_df))
# csv deki her veri için hesapla

#Sonuçları yeni CSV’de sakla
df.to_csv("sonuclar.csv", index=False)
df.to_csv("sonuclar.csv", index=False)
print("HMM sınıflandırması tamamlandı ve sonuclar.csv dosyasına kaydedildi.")


HMM sınıflandırması tamamlandı ve sonuclar.csv dosyasına kaydedildi.


In [29]:
from google.colab import files

files.download("sonuclar.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>